# Spectrum of a $\sin^4$ laser pulse

Builds the probe pulse used in the simulations and checks its spectrum two ways:

1. the damped Fourier transform from the `twoband` package,
2. NumPy's FFT.

Both must peak at the carrier frequency. No data files are needed.

In [ ]:
import numpy as np
from matplotlib import pyplot as plt

import twoband as tb

## Pulse parameters

In [ ]:
N_STEPS = 4500  # number of time steps
DT = 0.1  # time step (a.u.)
CARRIER = 163.10553149492998 / tb.HARTREE_TO_EV  # carrier frequency, the K edge (Hartree)
WIDTH = 100 / tb.HARTREE_TO_EV  # pulse duration (a.u.)

time = np.arange(N_STEPS) * DT
envelope = np.sin(np.pi * time / WIDTH) ** 4 * (time < WIDTH)
field = envelope * np.cos(CARRIER * time)

In [ ]:
fig, axs = plt.subplots(2, 1, figsize=(6, 5), sharex=True)
axs[0].plot(time, field)
axs[0].set_ylabel("E(t) (a.u.)")
axs[1].plot(time, field**2)
axs[1].set_ylabel(r"E(t)$^2$ (a.u.)")
axs[1].set_xlabel("Time (a.u.)")
axs[1].set_xlim(0, 50)
plt.show()

## Damped Fourier transform

$$F(\omega) = \int E(t)\, e^{i\omega t - \kappa t}\, dt, \qquad \kappa = -\ln(10^{-3}) / t_{max}$$

The damping brings the signal down to $10^{-3}$ at the end of the time window.

In [ ]:
energy = np.linspace(40, 300, 2000) / tb.HARTREE_TO_EV
kappa = -np.log(1e-3) / time[-1]
spectrum = tb.damped_ft(time, field, energy, kappa)

peak_ev = energy[np.argmax(np.abs(spectrum))] * tb.HARTREE_TO_EV
print(f"carrier: {CARRIER * tb.HARTREE_TO_EV:.2f} eV, damped FT peak: {peak_ev:.2f} eV")

## FFT

In [ ]:
field_fft = np.fft.rfft(field) * DT
omega_fft = np.fft.rfftfreq(N_STEPS, d=DT) * 2 * np.pi  # angular frequency (a.u.)

peak_fft_ev = omega_fft[np.argmax(np.abs(field_fft))] * tb.HARTREE_TO_EV
print(f"FFT peak: {peak_fft_ev:.2f} eV")

## Comparison

In [ ]:
plt.plot(energy * tb.HARTREE_TO_EV, np.abs(spectrum), lw=2, label="damped FT")
plt.plot(omega_fft * tb.HARTREE_TO_EV, np.abs(field_fft), "--", label="FFT")
plt.axvline(CARRIER * tb.HARTREE_TO_EV, color="k", lw=0.8, label="carrier")
plt.xlim(40, 300)
plt.xlabel("Energy (eV)")
plt.ylabel("|F(w)| (arb. units)")
plt.legend()
plt.show()